# Phase 2
Experiment 3: Single-Layer LoRA implementation for 13B llama-2 model using large dataset (10k samples) with LoRA Rank of 256



# Setup and Installations

setting up the environment and installs the required packages for fine-tuning a language model. It includes:

Importing necessary Python libraries and modules from transformers, datasets, peft, and trl.

Installing the latest version of trl to ensure compatibility and avoid known issues during training.

In [ ]:
!pip install -q accelerate> peft bitsandbytes transformers[torch] trl


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer


In [ ]:
!pip install -U trl # Upgrade to the latest trl version to solve the error.

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
!nvidia-smi


Fri Apr 11 22:52:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Configuration Settings for Fine-Tuning

This cell defines key configuration parameters for fine-tuning the LLaMA 2 7B Chat model using the Guanaco dataset:

In [ ]:
dataset_name = "timdettmers/openassistant-guanaco"
new_model = "Llama-2-7b-chat-finetune"
model_name = "NousResearch/Llama-2-7b-chat-hf"

# LoRA parameters
lora_r = 256
lora_alpha = 1
lora_dropout = 0.1

# Quantization settings
use_4bit = True
bnb_4bit_quant_type = "fp4"
bnb_4bit_compute_dtype = "bfloat16"
use_nested_quant = False

# Training arguments
output_dir = "./results"
num_train_epochs = 1
fp16 = False
bf16 = False
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 2
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 0
logging_steps = 25
max_seq_length = None
packing = False
device_map = {"": 0}


In [ ]:
# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

# Freezing Model Layers (Last-Layer Tuning)

This cell freezes all model layers to save memory and speed up training, then:

Unfreezes only the last transformer layer.

Converts its weights to float32 for stability.

Enables gradient updates only for that layer.

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Convert last layer to trainable dtype and enable gradients
last_layer_index = len(model.model.layers) - 1
for param in model.model.layers[last_layer_index].parameters():
    param.data = param.data.to(torch.float32)
    param.requires_grad = True

In [ ]:
# LoRA configuration for the last layer only
# Define LoRA configuration for only the supported submodules
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=[
        f"model.layers.{last_layer_index}.self_attn.q_proj",
        f"model.layers.{last_layer_index}.self_attn.k_proj",
        f"model.layers.{last_layer_index}.self_attn.v_proj",
        f"model.layers.{last_layer_index}.self_attn.o_proj",
        f"model.layers.{last_layer_index}.mlp.gate_proj",
        f"model.layers.{last_layer_index}.mlp.up_proj",
        f"model.layers.{last_layer_index}.mlp.down_proj"
    ],
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# Load dataset
dataset = load_dataset(dataset_name)

# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Fine-tuning setup
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    evaluation_strategy="no",
    save_strategy="no",
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    optim=optim,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
)

README.md:   0%|          | 0.00/395 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


openassistant_best_replies_train.jsonl:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

openassistant_best_replies_eval.jsonl:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9846 [00:00<?, ? examples/s]

...

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

Map:   0%|          | 0/9846 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/518 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Model Training with LoRA

This cell initializes and runs the training process:

Sets up the trainer using SFTTrainer with the fine-tuned model, tokenized dataset, and training configurations.

Applies the LoRA configuration to the model during training.

Starts the training process with the trainer.train() method.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_datasets["train"],
    args=training_args,
    peft_config=lora_config,  # Keep LoRA configuration
)


# Start training
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Truncating train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: asemelenawy16 (asemelenawy16-purdue-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
25,10.131900
50,10.301800
75,10.039000
100,9.849800
125,9.570900
150,9.057800
175,8.351400
200,8.345800
225,8.279300
250,8.479700


TrainOutput(global_step=4923, training_loss=3.311306267861741, metrics={'train_runtime': 4335.0939, 'train_samples_per_second': 2.271, 'train_steps_per_second': 1.136, 'total_flos': 1.780343731279872e+17, 'train_loss': 3.311306267861741})

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs

In [ ]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = "tell me an english sentence "
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] tell me an english sentence  [/INST]  The cat sat on the windowsill and juego with the butter:. Википедиya.
